# Обработка датасета | Dataset Processing

Этот раздел посвящен предварительной обработке аудиоданных: извлечению признаков (например, спектрограмм) и разделению данных на обучающую, валидационную и тестовую выборки.

This section is dedicated to audio data preprocessing: feature extraction (e.g., spectrograms) and splitting the data into training, validation, and test sets.

In [ ]:
from torch import nn

from ml.feature_extractors.splitter import DatasetSplitter
from ml.feature_extractors.pipeline import FeaturePipeline


def extract_features(input_dir:str,output_dir: str,extractor:nn.Module):

    splitter = DatasetSplitter(
        input_dir=input_dir,
    )

    pipeline = FeaturePipeline(
        extractor=extractor,
        output_dir=output_dir,
        target_sr=16000
    )

    pipeline.process_dataset(
        splitter=splitter,
        val_size=0.1,
        test_size=0.1
    )


In [ ]:
from ml.feature_extractors.mel import MelSpectrogramExtractor


input_dir = "data/in_the_wild"
output_dir = "data/in_the_wild/preprocessed"
extractor = MelSpectrogramExtractor(n_mels=128)
extract_features(input_dir=input_dir,output_dir=output_dir,extractor=extractor)

# Импорты и настройка параметров | Imports and Parameter Configuration

Здесь мы инициализируем библиотеки, загружаем архитектуры моделей (RCNN, GCNN, WavLM и др.) и устанавливаем глобальные параметры обучения, такие как скорость обучения (LR) и размер батча.

Here we initialize libraries, load model architectures (RCNN, GCNN, WavLM, etc.), and set global training parameters such as learning rate (LR) and batch size.

In [ ]:
import torch
from torch import nn

import multiprocessing

from ml.models.rcnn_model import RCNNSpoofDetector
from ml.models.gcnn_model import GCNNSpoofDetector
from ml.models.wavlm_detector import WavLMSpoofDetector
from ml.models.gwavlm_detector import GNNWavLMSpoofDetector
from ml.models.rwavlm_detector import LSTMWavLMSpoofDetector
from ml.models.mbwavlm_detector import MultiBranchWavLMSpoofDetector

from ml.datamodules.audiodatamodule import AudioDataModule
from ml.datamodules.asv_datamodule import ASV5DataModule

from ml.datasets.asv import ASVSpoof5Dataset
from ml.datasets.raw_audio import RawAudioDataset
from ml.datasets.preprocessed import PreprocessedDataset
from ml.datasets.collate import spectrogram_collate_fn

from ml.lightning_module import SpoofDetectorSystem

from ml.feature_extractors.mel import MelSpectrogramExtractor

import lightning as L
from lightning.pytorch.loggers import MLFlowLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar

import mlflow
import mlflow.pytorch


In [ ]:
def load_compat_checkpoint(checkpoint_path, model_arch: torch.nn.Module):
    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cuda"))
    state_dict = checkpoint["state_dict"]

    new_state_dict = {}
    for k, v in state_dict.items():
        if not k.startswith("model."):
            new_state_dict[f"model.{k}"] = v
        else:
            new_state_dict[k] = v

    system = SpoofDetectorSystem(model=model_arch)
    system.load_state_dict(new_state_dict, strict=False)
    return system

In [ ]:
max_epochs = 15
lr = 1e-3
bs = 32
pos_weight = 1.5

bar = TQDMProgressBar(refresh_rate=30)
multiprocessing.set_start_method('fork', force=True)
torch.set_float32_matmul_precision('medium')
mlflow.set_tracking_uri('http://localhost:5000')


In [ ]:
%load_ext autoreload
%autoreload 2
torch.cuda.set_per_process_memory_fraction(0.8, device=0)


In [ ]:
import gc
import torch


# Запустить сборщик мусора Python
gc.collect()

# Очистить кэш PyTorch
torch.cuda.empty_cache()

# Настройка функций | Function Configuration

Определение вспомогательных функций для конфигурации логгера MLFlow, настройки обратных вызовов (callbacks) и инкапсуляции логики циклов обучения и тестирования.

Definition of helper functions for MLFlow logger configuration, setting up callbacks, and encapsulating the logic for training and testing loops.

In [ ]:
def set_callbacks(
    model_name: str, refresh_rate: int = 50, every_n_train_steps: int = 200
) -> list:
    best_callback = ModelCheckpoint(
        dirpath="weights/best",
        monitor="val/eer",
        filename=f"{model_name}-{{epoch}}-{{val/eer:.4f}}",
        save_top_k=3,
        mode="min",
    )

    regular_callback = ModelCheckpoint(
        dirpath="weights/regular",
        every_n_train_steps=every_n_train_steps,
        filename=f"{model_name}-{{step}}",
        monitor="train/loss_step",
        save_top_k=5,
        mode="min",
    )

    early_stop = EarlyStopping(monitor="val/eer", patience=5, mode="min")

    bar = TQDMProgressBar(refresh_rate=refresh_rate)

    return [best_callback, regular_callback, early_stop, bar]

In [ ]:
def configure_logger(
    model_name: str,
    tags: dict,
    experiment_name: str = "Spoof_Detection_Project",
    tracking_uri: str = "http://localhost:5000",
) -> MLFlowLogger:
    mlf_logger = MLFlowLogger(
        experiment_name=experiment_name,
        run_name=f"{model_name.upper()}_Baseline",
        tracking_uri=tracking_uri,
        tags=tags,
    )

    mlflow.pytorch.autolog(log_datasets=True, checkpoint_monitor="val/eer")

    return mlf_logger

In [ ]:
def train_model(
    model_name: str,
    model_arch: object,
    tags: dict,
    data_module_class: type,
    dataset_class: type,
    lightning_module_class: type = SpoofDetectorSystem,
    experiment_name: str = "Spoof_Detection_Project",
    tracking_uri: str = "http://localhost:5000",
    lr: float = 1e-3,
    alpha: float = 0.5,
    gamma: float = 2.0,
    feature_extractor: nn.Module = None,
    max_epoch: int = 10,
    bs: int = 32,
    num_workers: int = 12,
    ckpt_path: str = None,
    run_test_after_train: bool = False,
    **datamodule_kwargs,
) -> L.Trainer:

    mlf_logger = configure_logger(
        model_name=model_name,
        tags=tags,
        experiment_name=experiment_name,
        tracking_uri=tracking_uri,
    )

    callbacks = set_callbacks(model_name=model_name)

    model = lightning_module_class(
        model=model_arch,
        lr=lr,
        alpha=alpha,
        gamma=gamma,
        feature_extractor=feature_extractor,
    )

    data_module = data_module_class(
        dataset_class=dataset_class,
        batch_size=bs,
        num_workers=num_workers,
        **datamodule_kwargs,
    )

    trainer = L.Trainer(
        max_epochs=max_epoch,
        accelerator="gpu",
        devices=1,
        logger=mlf_logger,
        callbacks=callbacks,
        precision="bf16-mixed",
        gradient_clip_val=1.0,
    )

    trainer.fit(
        model,
        datamodule=data_module,
        ckpt_path=ckpt_path,
    )

    if run_test_after_train:
        trainer.test(model=model, datamodule=data_module, ckpt_path="best")

    return trainer

In [ ]:
def test_model(
    model_name: str,
    ckpt_path: str,
    model_arch: object,
    data_module_class: type,
    dataset_class: type,
    tags: dict,
    feature_extractor: nn.Module = None,
    lightning_module_class: type = SpoofDetectorSystem,
    experiment_name: str = "Spoof_Detection_Project",
    tracking_uri: str = "http://localhost:5000",
    bs: int = 32,
    num_workers: int = 12,
    **datamodule_kwargs,
):
    mlf_logger = configure_logger(
        model_name=f"{model_name}_Test",
        tags=tags,
        experiment_name=experiment_name,
        tracking_uri=tracking_uri,
    )
    callbacks = set_callbacks(model_name=model_name)

    model = lightning_module_class.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        model=model_arch,
        feature_extractor=feature_extractor,
        strict=False,
    )

    data_module = data_module_class(
        dataset_class=dataset_class,
        batch_size=bs,
        num_workers=num_workers,
        **datamodule_kwargs,
    )

    trainer = L.Trainer(
        accelerator="gpu",
        devices=1,
        logger=mlf_logger,
        precision="bf16-mixed",
        callbacks=callbacks,
    )

    results = trainer.test(model, datamodule=data_module)

    return results

# Обучение RCNN на предобработанных признаках | RCNN Training on Preprocessed Features

Использование рекуррентно-сверточной нейронной сети (RCNN) для классификации спуфинга на основе заранее извлеченных мел-спектрограмм.

Using a Recurrent Convolutional Neural Network (RCNN) to classify spoofing based on pre-extracted Mel-spectrograms.

In [ ]:
train_model(
    model_name="rcnn",
    model_arch=RCNNSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=PreprocessedDataset,
    tags={"dataset": "fake or real","dataset_version": "preprocessed", "audio_type": "mel_spectrogram"},
    max_epoch=100,
    run_test_after_train=True,
    train_dir="ml/data/fake_or_real/preprocessed/training",
    val_dir="ml/data/fake_or_real/preprocessed/validation",
    test_dir="ml/data/fake_or_real/preprocessed/testing"
)

### RCNN: Датасет In The Wild | RCNN: In The Wild Dataset

Тестирование RCNN на данных 'In The Wild', содержащих реальные записи с различными акустическими условиями.

Testing RCNN on 'In The Wild' data, containing real-world recordings with diverse acoustic conditions.

In [ ]:
train_model(
    model_name="rcnn",
    model_arch=RCNNSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=PreprocessedDataset,
    tags={"dataset": "in the wild","dataset_version": "preprocessed", "audio_type": "mel_spectrogram"},
    max_epoch=100,
    run_test_after_train=True,
    train_dir="ml/data/in_the_wild/preprocessed/training",
    val_dir="ml/data/in_the_wild/preprocessed/validation",
    test_dir="ml/data/in_the_wild/preprocessed/testing",
    collate_fn=spectrogram_collate_fn,
)

### RCNN: Датасет ASVspoof5 | RCNN: ASVspoof5 Dataset

Оценка производительности RCNN на стандартном наборе данных ASVspoof5 для проверки обобщающей способности модели.

Evaluating RCNN performance on the standard ASVspoof5 dataset to verify the model's generalization ability.

In [ ]:
train_model(
    model_name="rcnn",
    model_arch=RCNNSpoofDetector(),
    feature_extractor=MelSpectrogramExtractor(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    max_epoch=3,
    run_test_after_train=True,
    root_dir="ml/data/asvspoof5",
)

### RCNN: Датасет ASVspoof5 | RCNN: ASVspoof5 Dataset

Тестирование производительности RCNN на стандартном наборе данных ASVspoof5 для проверки обобщающей способности модели.

Evaluating RCNN performance on the standard ASVspoof5 dataset to verify the model's generalization ability.

In [ ]:
test_model(
    model_name="rcnn",
    model_arch=RCNNSpoofDetector(),
    feature_extractor=MelSpectrogramExtractor(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/best/rcnn-epoch=18-val/eer=0.0025.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)

## GNN WavLM: Датасет In The Wild | GNN WavLM: In The Wild Dataset

Тестирование комбинации WavLM и графовых нейронных сетей (GNN) для обнаружения манипуляций в аудиопотоке на данных In The Wild.

Testing the combination of WavLM and Graph Neural Networks (GNN) for detecting audio stream manipulations on In The Wild data.

# Обучение GCNN на предобработанных признаках | GCNN Training on Preprocessed Features

Применение графовых сверточных сетей (GCNN) для анализа геометрических или структурных зависимостей в спектральных признаках мел-спектрограмм.

Applying Graph Convolutional Networks (GCNN) to analyze geometric or structural dependencies in Mel-spectrogram spectral features.

In [ ]:
train_model(
    model_name="gcnn",
    model_arch=GCNNSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=PreprocessedDataset,
    tags={"dataset": "fake or real","dataset_version": "preprocessed", "audio_type": "mel_spectrogram"},
    max_epoch=1,
    run_test_after_train=True,
    train_dir="ml/data/fake_or_real/preprocessed/training",
    val_dir="ml/data/fake_or_real/preprocessed/validation",
    test_dir="ml/data/fake_or_real/preprocessed/testing"
)

## GCNN: Датасет In The Wild | GCNN: In The Wild Dataset

Применение графовых сверточных сетей (GCNN) для анализа геометрических или структурных зависимостей в спектральных признаках датасета In The Wild.

Applying Graph Convolutional Networks (GCNN) to analyze geometric or structural dependencies in spectral features of the In The Wild dataset.

In [ ]:
train_model(
    model_name="gcnn",
    model_arch=GCNNSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=PreprocessedDataset,
    tags={"dataset": "in the wild","dataset_version": "preprocessed", "audio_type": "mel_spectrogram"},
    max_epoch=100,

    run_test_after_train=True,
    train_dir="ml/data/in_the_wild/preprocessed/training",
    val_dir="ml/data/in_the_wild/preprocessed/validation",
    test_dir="ml/data/in_the_wild/preprocessed/testing",
    collate_fn=spectrogram_collate_fn,
)

In [ ]:
test_model(
    model_name="gcnn",
    model_arch=GCNNSpoofDetector(),
    feature_extractor=MelSpectrogramExtractor(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/best/gcnn-epoch=6-val/eer=0.0032.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)

# WavLM: Использование сырого аудио | WavLM: Raw Audio Processing

Обучение детектора на базе предобученной модели WavLM, которая работает напрямую с необработанным аудиосигналом для извлечения контекстных представлений.

Training a detector based on the pre-trained WavLM model, which operates directly on raw audio signals to extract contextual representations.

In [ ]:
train_model(
    model_name="wavlm",
    model_arch=WavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    max_epoch=10,
    bs=16,
    run_test_after_train=True,
    root_dir="ml/data/in_the_wild",
)

## WavLM: Датасет Fake or Real | WavLM: Fake or Real Dataset

Обучение детектора на базе предобученной модели WavLM на датасете Fake or Real.

Training a detector based on the pre-trained WavLM model on the Fake or Real dataset.

In [ ]:
train_model(
    model_name="wavlm",
    model_arch=WavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    tags={
        "dataset": "fake or real",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    max_epoch=1,
    run_test_after_train=True,
    ckpt_path="weights/regular/wavlm-step=200.ckpt",
    train_dir="ml/data/fake_or_real/training",
    val_dir="ml/data/fake_or_real/validation",
    test_dir="ml/data/fake_or_real/testing",
)

In [ ]:
test_model(
    model_name="wavlm",
    model_arch=WavLMSpoofDetector(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/best/wavlm-epoch=9-val/eer=0.0154.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)

# GNN WavLM: Датасет In The Wild | GNN WavLM: In The Wild Dataset

Обучение комбинации WavLM и графовых нейронных сетей (GNN) для обнаружения манипуляций в аудиопотоке на данных In The Wild.

Training the combination of WavLM and Graph Neural Networks (GNN) for detecting audio stream manipulations on In The Wild data.

In [ ]:

train_model(
    model_name="gwavlm",
    model_arch=GNNWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    ckpt_path="weights/best/gwavlm-epoch=5-val/eer=0.0073.ckpt",
    gamma=1,
    bs=16,
    max_epoch=10,
    lr=5e-5,
    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    run_test_after_train=True,
    root_dir="ml/data/in_the_wild",
)

## GNN WavLM: Датасет In The Wild | GNN WavLM: In The Wild Dataset

Тестирование комбинации WavLM и графовых нейронных сетей (GNN) для обнаружения манипуляций в аудиопотоке на данных In The Wild.

Testing the combination of WavLM and Graph Neural Networks (GNN) for detecting audio stream manipulations on In The Wild data.

In [ ]:

test_model(
    model_name="gwavlm",
    model_arch=GNNWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    ckpt_path="weights/best/gwavlm-epoch=5-val/eer=0.0073.ckpt",
    bs=16,

    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    root_dir="ml/data/in_the_wild",
)

In [ ]:
test_model(
    model_name="gwavlm",
    model_arch=GNNWavLMSpoofDetector(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/best/gwavlm-epoch=5-val/eer=0.0073.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)

# LSTM (rWavLM): Рекуррентный анализ | LSTM (rWavLM): Recurrent Analysis

Комбинация WavLM и LSTM слоев для захвата долгосрочных временных зависимостей в аудиозаписях.

Combining WavLM and LSTM layers to capture long-term temporal dependencies in audio recordings.

In [ ]:

test_model(
    model_name="rwavlm",
    model_arch=LSTMWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    ckpt_path="weights/best/rwavlm-epoch=5-val/eer=0.0436.ckpt",
    bs=32,

    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    root_dir="ml/data/in_the_wild",
)

In [ ]:
train_model(
    model_name="rwavlm",
    model_arch=LSTMWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    lr=5e-5,
    tags={
        "dataset": "fake or real",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    max_epoch=1,
    run_test_after_train=True,
    root_dir="ml/data/fake_or_real",
)

## LSTM (rWavLM): Датасет In The Wild | LSTM (rWavLM): In The Wild Dataset

Применение рекуррентной архитектуры rWavLM для обработки записей из набора данных In The Wild.

Applying the rWavLM recurrent architecture to process recordings from the In The Wild dataset.

In [ ]:
train_model(
    model_name="rwavlm",
    model_arch=LSTMWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    lr=5e-5,
    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    bs=16,
    max_epoch=10,
    run_test_after_train=True,
    root_dir="ml/data/in_the_wild",
)

## LSTM (rWavLM): Датасет ASVspoof5 | LSTM (rWavLM): ASVspoof5 Dataset

Стресс-тестирование рекуррентной модели на данных ASVspoof5, являющихся эталоном в индустрии.

Stress-testing the recurrent model on ASVspoof5 data, which serves as an industry benchmark.

In [ ]:
train_model(
    model_name="rwavlm",
    model_arch=LSTMWavLMSpoofDetector(),
    lr=5e-5,
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    max_epoch=1,
    run_test_after_train=True,
    root_dir="ml/data/asvspoof5",
)

In [ ]:
test_model(
    model_name="rwavlm",
    model_arch=LSTMWavLMSpoofDetector(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/best/rwavlm-epoch=5-val/eer=0.0436.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)

# Multi-Branch WavLM (SOTA): Продвинутая архитектура | Multi-Branch WavLM (SOTA): Advanced Architecture

Использование многоветвевой структуры для агрегации признаков с разных уровней модели WavLM, что является современным стандартом (SOTA) в детектировании спуфинга.

Using a multi-branch structure to aggregate features from different levels of the WavLM model, representing the state-of-the-art (SOTA) in spoofing detection.

In [ ]:
train_model(
    model_name="mbwavlm",
    model_arch=MultiBranchWavLMSpoofDetector(),
    data_module_class=AudioDataModule,
    dataset_class=RawAudioDataset,
    lr=5e-5,
    tags={
        "dataset": "in the wild",
        "dataset_version": "raw",
        "audio_type": "wav",
    },
    bs=16,
    max_epoch=5,
    run_test_after_train=True,
    root_dir="ml/data/in_the_wild",
)

## Multi-Branch WavLM: Датасет ASVspoof5 | Multi-Branch WavLM: ASVspoof5 Dataset

Использование SOTA архитектуры Multi-Branch для достижения наилучших результатов на данных ASVspoof5.

Using the SOTA Multi-Branch architecture to achieve the best results on ASVspoof5 data.

In [ ]:
train_model(
    model_name="mbwavlm",
    model_arch=MultiBranchWavLMSpoofDetector(n_last_layers=0),
    lr=5e-5,
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    bs=24,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    max_epoch=1,
    run_test_after_train=True,
    root_dir="ml/data/asvspoof5",
)

In [ ]:
test_model(
    model_name="mbwavlm",
    model_arch=MultiBranchWavLMSpoofDetector(),
    data_module_class=ASV5DataModule,
    dataset_class=ASVSpoof5Dataset,
    ckpt_path="weights/regular/mbwavlm-step=10600.ckpt",
    bs=16,
    tags={"dataset": "asv", "dataset_version": "raw", "audio_type": "flac"},
    root_dir="ml/data/asvspoof5",
)